# Lab 5.2: Coordinator-Subagent Patterns

**What you'll build:** A coordinator-subagent system -- and learn why monolithic agents with too many tools fail.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- monolithic agent with 12+ tools makes wrong tool choices | 5 min |
| 2 | The Right Way -- coordinator dispatches to focused subagents | 10 min |
| 3 | Your Turn -- add a new subagent to the system | 5 min |
| 4 | Stress Test -- verify context isolation between subagents | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You are building a customer service system that needs to:
1. **Look up customer info** (name, account status, tier)
2. **Check order status** (order ID, shipping status, delivery estimate)
3. **Process actions** (refunds, exchanges, escalations)

First, we'll try a monolithic agent with all tools. Then we'll build the coordinator-subagent pattern.

In [ ]:
# Simulated backend data
CUSTOMERS = {
    "C001": {"name": "Alice Johnson", "tier": "premium", "status": "active", "email": "alice@example.com"},
    "C002": {"name": "Bob Smith", "tier": "basic", "status": "active", "email": "bob@example.com"},
}

ORDERS = {
    "ORD-100": {"customer_id": "C001", "status": "shipped", "total": 89.99, "items": ["Widget Pro", "Gadget X"], "tracking": "TRK123"},
    "ORD-101": {"customer_id": "C001", "status": "delivered", "total": 24.99, "items": ["Cable Set"], "tracking": "TRK456"},
    "ORD-102": {"customer_id": "C002", "status": "processing", "total": 149.99, "items": ["Mega Bundle"], "tracking": None},
}

REFUND_LOG = []

# All the tools a monolithic agent would need
ALL_TOOLS = [
    {"name": "lookup_customer", "description": "Look up customer by ID", "input_schema": {"type": "object", "properties": {"customer_id": {"type": "string"}}, "required": ["customer_id"]}},
    {"name": "search_customer_by_email", "description": "Search for customer by email", "input_schema": {"type": "object", "properties": {"email": {"type": "string"}}, "required": ["email"]}},
    {"name": "get_customer_orders", "description": "Get all orders for a customer", "input_schema": {"type": "object", "properties": {"customer_id": {"type": "string"}}, "required": ["customer_id"]}},
    {"name": "get_order_details", "description": "Get details for a specific order", "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}},
    {"name": "track_shipment", "description": "Get tracking info for an order", "input_schema": {"type": "object", "properties": {"tracking_number": {"type": "string"}}, "required": ["tracking_number"]}},
    {"name": "estimate_delivery", "description": "Estimate delivery date", "input_schema": {"type": "object", "properties": {"tracking_number": {"type": "string"}}, "required": ["tracking_number"]}},
    {"name": "process_refund", "description": "Process a refund for an order", "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}, "amount": {"type": "number"}, "reason": {"type": "string"}}, "required": ["order_id", "amount", "reason"]}},
    {"name": "process_exchange", "description": "Process an exchange for an order", "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}, "new_item": {"type": "string"}}, "required": ["order_id", "new_item"]}},
    {"name": "escalate_to_supervisor", "description": "Escalate to human supervisor", "input_schema": {"type": "object", "properties": {"reason": {"type": "string"}, "context": {"type": "string"}}, "required": ["reason", "context"]}},
    {"name": "send_email", "description": "Send email to customer", "input_schema": {"type": "object", "properties": {"to": {"type": "string"}, "subject": {"type": "string"}, "body": {"type": "string"}}, "required": ["to", "subject", "body"]}},
    {"name": "update_customer_notes", "description": "Add notes to customer record", "input_schema": {"type": "object", "properties": {"customer_id": {"type": "string"}, "note": {"type": "string"}}, "required": ["customer_id", "note"]}},
    {"name": "check_inventory", "description": "Check product inventory", "input_schema": {"type": "object", "properties": {"product_name": {"type": "string"}}, "required": ["product_name"]}},
]

print(f"Monolithic agent: {len(ALL_TOOLS)} tools")
print("Tools:", [t['name'] for t in ALL_TOOLS])

In [ ]:
def execute_service_tool(name, tool_input):
    """Execute a customer service tool."""
    if name == "lookup_customer":
        cust = CUSTOMERS.get(tool_input["customer_id"])
        return json.dumps(cust if cust else {"error": "Customer not found"})
    elif name == "search_customer_by_email":
        for cid, cust in CUSTOMERS.items():
            if cust["email"] == tool_input["email"]:
                return json.dumps({"customer_id": cid, **cust})
        return json.dumps({"error": "No customer found with that email"})
    elif name == "get_customer_orders":
        orders = {oid: o for oid, o in ORDERS.items() if o["customer_id"] == tool_input["customer_id"]}
        return json.dumps(orders if orders else {"error": "No orders found"})
    elif name == "get_order_details":
        order = ORDERS.get(tool_input["order_id"])
        return json.dumps(order if order else {"error": "Order not found"})
    elif name == "track_shipment":
        return json.dumps({"tracking": tool_input["tracking_number"], "status": "In transit", "location": "Distribution center"})
    elif name == "estimate_delivery":
        return json.dumps({"estimated_delivery": "2025-02-15", "confidence": "high"})
    elif name == "process_refund":
        REFUND_LOG.append(tool_input)
        return json.dumps({"status": "approved", "refund_id": "REF-001", "amount": tool_input["amount"]})
    elif name == "process_exchange":
        return json.dumps({"status": "initiated", "exchange_id": "EXC-001"})
    elif name == "escalate_to_supervisor":
        return json.dumps({"status": "escalated", "ticket_id": "ESC-001"})
    elif name == "send_email":
        return json.dumps({"status": "sent", "to": tool_input["to"]})
    elif name == "update_customer_notes":
        return json.dumps({"status": "updated"})
    elif name == "check_inventory":
        return json.dumps({"product": tool_input["product_name"], "in_stock": True, "quantity": 42})
    return json.dumps({"error": f"Unknown tool: {name}"})

print("Tool executor ready.")

---

## Phase 1: The Wrong Approach

Give a single agent all 12 tools and ask it to handle a customer request. Watch how tool selection becomes unreliable.

In [ ]:
def run_monolithic_agent(user_message):
    """Run a single agent with ALL tools."""
    messages = [{"role": "user", "content": user_message}]
    tool_calls_made = []
    
    for _ in range(10):  # safety cap
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=ALL_TOOLS,
            messages=messages
        )
        
        if response.stop_reason == "end_turn":
            final = "".join(b.text for b in response.content if hasattr(b, "text"))
            return final, tool_calls_made
        
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                tool_calls_made.append(block.name)
                result = execute_service_tool(block.name, block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
        if tool_results:
            messages.append({"role": "user", "content": tool_results})
    
    return "(hit safety cap)", tool_calls_made


# Run the monolithic agent
query = "Customer C001 wants to know the status of order ORD-100 and whether they can get a refund for order ORD-101."
mono_result, mono_tools = run_monolithic_agent(query)

print(f"Monolithic agent used {len(mono_tools)} tool calls:")
for i, t in enumerate(mono_tools, 1):
    print(f"  {i}. {t}")
print(f"\nResponse preview: {mono_result[:200]}...")

### The Problem with Monolithic Agents

With 12 tools, the agent may:
- Call unnecessary tools (e.g., `check_inventory` when asked about order status)
- Miss the right tool when many similar tools exist
- Take more steps than necessary because it is exploring the tool space

Research shows tool selection accuracy degrades with 10+ tools. The fix: split into focused subagents.

---

## Phase 2: The Right Approach — Coordinator-Subagent Pattern

Build a coordinator that dispatches to specialized subagents, each with 3-4 focused tools.

In [ ]:
# Define scoped tool sets for each subagent
CUSTOMER_TOOLS = [
    t for t in ALL_TOOLS if t["name"] in ["lookup_customer", "search_customer_by_email", "update_customer_notes"]
]

ORDER_TOOLS = [
    t for t in ALL_TOOLS if t["name"] in ["get_customer_orders", "get_order_details", "track_shipment", "estimate_delivery"]
]

ACTION_TOOLS = [
    t for t in ALL_TOOLS if t["name"] in ["process_refund", "process_exchange", "escalate_to_supervisor", "send_email"]
]

print(f"Customer subagent: {len(CUSTOMER_TOOLS)} tools - {[t['name'] for t in CUSTOMER_TOOLS]}")
print(f"Order subagent:    {len(ORDER_TOOLS)} tools - {[t['name'] for t in ORDER_TOOLS]}")
print(f"Action subagent:   {len(ACTION_TOOLS)} tools - {[t['name'] for t in ACTION_TOOLS]}")

In [ ]:
def run_subagent(task_prompt, tools, label="Subagent"):
    """
    Run a focused subagent with scoped tools.
    
    KEY: The subagent starts with a FRESH context.
    It only knows what the coordinator tells it via task_prompt.
    """
    messages = [{"role": "user", "content": task_prompt}]
    tool_calls = []
    
    for _ in range(10):
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=tools,
            messages=messages
        )
        
        if response.stop_reason == "end_turn":
            final = "".join(b.text for b in response.content if hasattr(b, "text"))
            print(f"  [{label}] Completed with {len(tool_calls)} tool call(s): {tool_calls}")
            return final
        
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                tool_calls.append(block.name)
                result = execute_service_tool(block.name, block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
        if tool_results:
            messages.append({"role": "user", "content": tool_results})
    
    return "(subagent hit safety cap)"


def run_coordinator(user_request):
    """
    Coordinator: decomposes the request and dispatches to subagents.
    
    The coordinator does NOT call service tools directly.
    It analyzes the request and dispatches to focused subagents.
    """
    print(f"=== COORDINATOR ===")
    print(f"Request: {user_request}\n")
    
    # Step 1: Customer lookup (always needed)
    print("Step 1: Dispatching Customer Subagent...")
    customer_info = run_subagent(
        f"Look up customer C001. Return their name, tier, and status in a brief summary.",
        CUSTOMER_TOOLS,
        "Customer"
    )
    print(f"  Result: {customer_info[:100]}\n")
    
    # Step 2: Order lookup (pass ONLY the needed context)
    print("Step 2: Dispatching Order Subagent...")
    order_info = run_subagent(
        f"Get the status of orders ORD-100 and ORD-101 for customer C001. "
        f"For each order, report: order ID, status, total, and items. "
        f"Return a brief structured summary.",
        ORDER_TOOLS,
        "Order"
    )
    print(f"  Result: {order_info[:150]}\n")
    
    # Step 3: Action (refund check -- pass context from previous steps)
    print("Step 3: Dispatching Action Subagent...")
    action_result = run_subagent(
        f"A premium customer (C001, Alice Johnson) wants a refund for order ORD-101. "
        f"The order is delivered, total was $24.99, items: Cable Set. "
        f"Process the refund for the full amount with reason 'customer request'. "
        f"Confirm the refund was processed.",
        ACTION_TOOLS,
        "Action"
    )
    print(f"  Result: {action_result[:150]}\n")
    
    # Coordinator synthesizes
    print("=== COORDINATOR SYNTHESIS ===")
    summary = (
        f"Customer: {customer_info[:80]}\n"
        f"Orders: {order_info[:120]}\n"
        f"Action: {action_result[:120]}"
    )
    return summary


coordinator_result = run_coordinator(query)
print(f"\n=== FINAL RESULT ===")
print(coordinator_result)

### Key Observations

1. **Each subagent used only its scoped tools** -- no confusion between customer lookup and order lookup
2. **Each subagent had fresh context** -- the Order subagent did not see the Customer subagent's conversation
3. **The coordinator passed explicit context** -- when the Action subagent needed customer info, the coordinator included it in the prompt
4. **Communication flowed through the coordinator** -- subagents never talked to each other directly

---

## Phase 3: Your Turn

Add an **Inventory subagent** that can check product availability. Then modify the coordinator to dispatch to it.

**Scenario:** Customer C002 wants to know if "Mega Bundle" is in stock for a potential reorder.

In [ ]:
# =============================================================
# TODO: Define INVENTORY_TOOLS and dispatch an inventory subagent
# =============================================================
#
# 1. Create INVENTORY_TOOLS (subset of ALL_TOOLS with check_inventory)
# 2. Run a subagent with a clear task prompt
# 3. The subagent should check inventory for "Mega Bundle"
#    and return availability info

# INVENTORY_TOOLS = [t for t in ALL_TOOLS if t["name"] in ["check_inventory"]]

# inventory_result = run_subagent(
#     "Check if 'Mega Bundle' is currently in stock. Report the product name, availability, and quantity.",
#     INVENTORY_TOOLS,
#     "Inventory"
# )
# print(f"Inventory result: {inventory_result}")

print("TODO: Complete the inventory subagent above.")

---

## Phase 4: Stress Test — Context Isolation

Let's verify that subagents truly do NOT inherit coordinator context.

In [ ]:
# Test: Does the order subagent know about the customer lookup?
# If context is properly isolated, asking about customer info should fail.

print("=== Context Isolation Test ===")
print("Asking the Order subagent about customer details (it should NOT know)...\n")

isolation_test = run_subagent(
    "What is the customer's name and email? (Do not make assumptions -- only use your tools.)",
    ORDER_TOOLS,
    "Order (isolation test)"
)

print(f"\nOrder subagent response: {isolation_test}")
print()

# The order subagent should NOT know Alice's name or email
# because it was never passed that context
if "alice" in isolation_test.lower() and "alice" not in "":
    print("WARNING: Subagent appears to know customer info it should not have.")
    print("This suggests context leakage -- check your implementation.")
else:
    print("PASSED -- Order subagent does not have customer info.")
    print("Context isolation is working correctly.")

### Key Takeaways

1. **Too many tools degrades selection.** 12 tools on one agent is worse than 3-4 tools on focused subagents.
2. **Subagents do NOT inherit context.** Each subagent starts fresh. The coordinator must explicitly pass needed information.
3. **All communication flows through the coordinator.** Subagents never talk to each other directly.
4. **The coordinator synthesizes.** It holds the overall task context and combines subagent results.